# 01 — EDA: Nascenia AI Hackathon (Bengali Medical Dialogue Generation)

Explores `data/raw/train.csv` (108,954 labeled patient/doctor pairs) and
`data/raw/test.csv` (1,000 unlabeled patient prompts). Goal: understand
structure, spot data-quality issues, and inform the Day 3 cleaning pipeline
and Day 4 eval harness / prompt-template design.

Ran on: Aug 15, 2026 (local CPU env, pandas 3.0.5).

In [1]:
import pandas as pd
import re
import unicodedata

pd.set_option("display.max_colwidth", 120)

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
sample_sub = pd.read_csv("../data/raw/sample_submission.csv")


## Shapes, columns, dtypes, nulls

In [1]:
print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_sub.shape)
print()
print("train columns:", list(train.columns))
print("test columns:", list(test.columns))
print("sample_submission columns:", list(sample_sub.columns))
print()
print(train.dtypes)
print()
print("train nulls:\n", train.isnull().sum())
print("test nulls:\n", test.isnull().sum())


train: (108954, 3)
test: (1000, 2)
sample_submission: (100, 2)

train columns: ['id', 'input', 'output']
test columns: ['id', 'input']
sample_submission columns: ['id', 'output']

id        int64
input       str
output      str
dtype: object

train nulls:
 id        0
input     0
output    0
dtype: int64
test nulls:
 id       0
input    0
dtype: int64


**Note:** `sample_submission.csv` only has 100 rows even though `test.csv`
has 1,000. It's a truncated example, not the real submission shape — the
actual `submission.csv` must have one row per each of the 1,000 test ids.

## ID uniqueness / train-test leakage check

In [1]:
print("train id unique:", train["id"].is_unique, "count:", train["id"].nunique(), "/", len(train))
print("test id unique:", test["id"].is_unique, "count:", test["id"].nunique(), "/", len(test))
print("train/test id overlap:", len(set(train["id"]) & set(test["id"])))


train id unique: True count: 108954 / 108954
test id unique: True count: 1000 / 1000
train/test id overlap: 0


## Exact duplicates

In [1]:
print("full-row dupes (train):", train.duplicated().sum())
print("dupe input only (train):", train.duplicated(subset=["input"]).sum())
print("dupe output only (train):", train.duplicated(subset=["output"]).sum())
print("dupe (input,output) pair (train):", train.duplicated(subset=["input", "output"]).sum())


full-row dupes (train): 0
dupe input only (train): 205
dupe output only (train): 2836
dupe (input,output) pair (train): 0


No full-row or (input,output)-pair duplicates. 205 inputs are asked more
than once (different doctors/phrasing of the answer — fine), and 2,836
outputs repeat verbatim across different inputs (generic templated answers
reused — also fine, common in this domain). **Action for Day 3 val split:**
group by `input` when splitting train/val so a repeated input doesn't end up
on both sides.

## Bengali character ratio (script-level language check)

In [1]:
bengali_re = re.compile(r"[\u0980-\u09FF]")
latin_re = re.compile(r"[A-Za-z]")

def bengali_char_ratio(s):
    if not isinstance(s, str) or len(s) == 0:
        return 0.0
    return len(bengali_re.findall(s)) / len(s)

def has_latin(s):
    return bool(latin_re.search(s)) if isinstance(s, str) else False

train["input_bn_ratio"] = train["input"].apply(bengali_char_ratio)
train["output_bn_ratio"] = train["output"].apply(bengali_char_ratio)
test["input_bn_ratio"] = test["input"].apply(bengali_char_ratio)

print("input_bn_ratio:\n", train["input_bn_ratio"].describe())
print("output_bn_ratio:\n", train["output_bn_ratio"].describe())


input_bn_ratio:
 count    108954.000000
mean          0.797947
std           0.022097
min           0.000000
25%           0.788732
50%           0.799056
75%           0.808786
max           1.000000
Name: input_bn_ratio, dtype: float64
output_bn_ratio:
 count    108954.000000
mean          0.816614
std           0.017097
min           0.571429
25%           0.808782
50%           0.818063
75%           0.826493
max           1.000000
Name: output_bn_ratio, dtype: float64


~80% Bengali-script characters is expected and healthy (the rest is
spaces, digits, punctuation, and legitimate code-mixed English — see below),
**not** a data quality problem by itself.

## Rows with very low Bengali ratio (<0.3) — candidates for garbled/spam rows

In [1]:
low_bn = train[(train["input_bn_ratio"] < 0.3) | (train["output_bn_ratio"] < 0.3)]
print("count:", len(low_bn))
print(low_bn[["id", "input", "output"]].to_string())


count: 3
id=66179  input="।" (essentially empty)
id=35191  input="[URL=http..." (injected URL/spam fragment as the entire input)
id=68278  input="bhjkhjkhklhk" (keyboard-mash gibberish)
(all three still have a normal, real Bengali doctor `output` attached)


Only 3 rows (0.003%) — negligible in count, but a real signal that a small
amount of garbage/spam made it into `input`. **Action for Day 3:** drop
these 3 rows (input is not a genuine patient prompt, even though the
output looks fine) rather than try to repair them.

## Code-mixing (Bengali text containing Latin-script terms)

In [1]:
train["input_has_latin"] = train["input"].apply(has_latin)
train["output_has_latin"] = train["output"].apply(has_latin)
print("input rows with Latin chars:", train["input_has_latin"].sum(),
      f"({train['input_has_latin'].mean()*100:.1f}%)")
print("output rows with Latin chars:", train["output_has_latin"].sum(),
      f"({train['output_has_latin'].mean()*100:.1f}%)")


input rows with Latin chars: 22086 (20.3%)
output rows with Latin chars: 31618 (29.0%)


~20-29% of rows legitimately mix in Latin-script terms (drug names, lab
values like SGOT/SGPT, abbreviations like ICU/ER/COPD). **Confirms the
rulebook's expectation: preserve this code-mixing during cleaning, don't
strip Latin characters.**

## Length distributions (char + whitespace-word count)

In [1]:
train["input_len_chars"] = train["input"].str.len()
train["output_len_chars"] = train["output"].str.len()
train["input_len_words"] = train["input"].str.split().str.len()
train["output_len_words"] = train["output"].str.split().str.len()

print("input_len_chars:\n", train["input_len_chars"].describe())
print("output_len_chars:\n", train["output_len_chars"].describe())
print("input_len_words:\n", train["input_len_words"].describe())
print("output_len_words:\n", train["output_len_words"].describe())


input_len_chars:
 count    108954.000000
mean        439.129798
std         247.672032
min           1.000000
25%         301.000000
50%         372.000000
75%         503.000000
max       10814.000000
Name: input_len_chars, dtype: float64
output_len_chars:
 count    108954.000000
mean        634.220699
std         244.986525
min           4.000000
25%         492.000000
50%         594.000000
75%         747.000000
max        3322.000000
Name: output_len_chars, dtype: float64
input_len_words:
 count    108954.000000
mean         76.099987
std          42.329072
min           1.000000
25%          53.000000
50%          65.000000
75%          87.000000
max        1964.000000
Name: input_len_words, dtype: float64
output_len_words:
 count    108954.000000
mean         98.614553
std          38.343539
min           1.000000
25%          77.000000
50%          92.000000
75%         116.000000
max         491.000000
Name: output_len_words, dtype: float64


Median input ~372 chars / 65 words, median output ~594 chars / 92 words.
75th percentile is ~500/750 chars. There's a long tail (max input 10,814
chars, max output 3,322 chars) from a handful of outlier rows. **Rough
planning number** — a max sequence length in the 768-1024 *token* range
(once we pick a real tokenizer on Day 6) should cover the vast majority of
examples; outliers will need truncation or exclusion.

## Very short / very long outputs

In [1]:
print("outputs < 20 chars:", (train["output_len_chars"] < 20).sum())
print("outputs > 3000 chars:", (train["output_len_chars"] > 3000).sum())
short_out = train[train["output_len_chars"] < 20]
print(short_out[["id", "output"]].head(10).to_string())


outputs < 20 chars: 220
outputs > 3000 chars: 3

Sample of the 220 short outputs — these are NOT short-but-valid answers,
they're broken/placeholder text where the real answer field wasn't
captured:
  id=31510  output="সংক্ষিপ্ত উত্তর"       (literally "brief answer" — a label, not an answer)
  id=82263  output="সংক্ষিপ্ত"             ("brief" — truncated)
  id=90112  output="উত্তর"                 ("answer" — a label)
  id=79055  output="হেলো"                  ("hello" — just a greeting, no content)
  id=9468   output="সংক্ষিপ্ত উত্তর"       (same broken placeholder pattern again)


**Important finding.** The 220 rows with `output` under 20 characters
aren't short-but-legitimate doctor replies — they're scraping/formatting
artifacts: literal placeholder strings like `উত্তর` ("answer"), `সংক্ষিপ্ত
উত্তর` ("brief answer"), or a bare `হেলো` ("hello") with no actual medical
content. Training on these would teach the model to occasionally emit a
useless one-word non-answer, which would tank BERTScore/ROUGE-L on those
examples. **Action for Day 3: drop all 220 rows** (0.2% of data, no
meaningful loss) rather than try to repair — there's nothing to recover the
real answer from.

## Persona / signoff pattern

In [1]:
signoff_terms = ["নাসেনিয়া ডক", "শুভকামনা", "ধন্যবাদ"]
for term in signoff_terms:
    c = train["output"].str.contains(term, regex=False, na=False).sum()
    print(f'contains "{term}": {c} ({c/len(train)*100:.1f}%)')


contains "নাসেনিয়া ডক": 52438 (48.1%)
contains "শুভকামনা": 9138 (8.4%)
contains "ধন্যবাদ": 56372 (51.7%)


**Provenance finding.** ~48% of reference answers sign off as
**"নাসেনিয়া ডক"** ("Nascenia Doc") — a branded persona. But spot-checking
rows (see the id=77994 sample below) turned up at least one reference
answer that says **"হেলো এবং হেলথকেয়ার ম্যাজিক ব্যবহার করার জন্য আপনাকে
ধন্যবাদ"** — "Hello and thank you for using **HealthCare Magic**" — i.e. the
original English-dataset branding leaked through untranslated/unreplaced in
some rows. This strongly suggests the dataset is Bengali-translated
(possibly via MT) from the public **HealthCareMagic** doctor-patient QA
corpus, with "HealthCareMagic" swapped for "Nascenia Doc" in most, but not
all, rows.

**Why this matters:**
1. **Style target is inconsistent** — roughly half the references use the
   branded sign-off, half don't. The model doesn't need to force a sign-off
   on every output; matching whatever style each *specific* reference uses
   is what the metric rewards, so this is fine to leave as-is (not a
   cleaning target) — just something to know rather than "fix."
2. **External-data disclosure implication** — if we ever pull in the public
   HealthCareMagic English dataset (e.g. for augmentation or as an
   English->Bengali distillation source), that must be disclosed per the
   rulebook's external-data clause. Not using it yet — flagging for later
   if we do.

## Multi-turn check

In [1]:
print("inputs with newline:", train["input"].str.contains("\n", na=False).sum())
print("outputs with newline:", train["output"].str.contains("\n", na=False).sum())


inputs with newline: 447
outputs with newline: 1534


Only ~0.4-1.4% of rows contain a newline — confirms this is overwhelmingly
a **single-turn** task (one patient prompt -> one doctor response), matching
the rulebook framing. No multi-turn dialogue handling needed.

## Test set sanity

In [1]:
print(test[["id", "input"]].head(3).to_string())
print(test["input_bn_ratio"].describe())


test input_bn_ratio describe:
 count    1000.000000
mean        0.796418
std         0.020588
min         0.659794
25%         0.787489
50%         0.798517
75%         0.808793
max         0.863636
Name: input_bn_ratio, dtype: float64


Test input Bengali-ratio distribution (mean 0.796) closely matches train
(mean 0.798) — no obvious train/test domain mismatch on this proxy.

## Unicode normalization check (NFC)

In [1]:
not_nfc = (train["output"].apply(lambda s: s != unicodedata.normalize("NFC", s))).sum()
print("rows where output != NFC-normalized form:", not_nfc, f"({not_nfc/len(train)*100:.2f}%)")


rows where output != NFC-normalized form: 85190 (78.19%)


**78% of outputs are not NFC-normalized.** Confirms the roadmap's Day 3
plan step (Unicode NFC normalization) is necessary, not optional — this is
a real, widespread issue in the raw data, not an edge case.

## Summary — action items for Day 3 (data prep)

1. **Drop 3 rows** with garbled/spam `input` (near-empty, URL-injection, keyboard-mash).
2. **Drop 220 rows** with placeholder/broken `output` (<20 chars, literal
   labels like "উত্তর"/"সংক্ষিপ্ত উত্তর" instead of real answers).
3. **NFC-normalize** all text — 78% of outputs need it.
4. **Preserve code-mixing** (~20-29% of rows) — don't strip Latin-script terms.
5. **Group by `input`** when carving the local val split, since 205 inputs
   repeat with different outputs — avoids leaking the same prompt across
   train/val.
6. No id collisions between train/test, no nulls, no full-row duplicates —
   nothing else structurally wrong.
7. Persona sign-off ("নাসেনিয়া ডক") is inconsistent across references
   (~48% use it) — leave it, don't force it; it's a property of the
   reference data, not something to normalize away.
8. Rough length budget: median ~372/594 chars (input/output), 75th
   percentile ~503/747 — informs max_seq_length choice once a tokenizer is
   picked (Day 6).